# Classification: Trial-Level Bias Score (TL-BS)

Based on Zvielli, Bernstein & Koster (2015), attentional bias is treated as a dynamic signal rather than a stable trait. Instead of averaging fixation bias across all trials, we extract parameters that describe the shape and dynamics of the bias signal across the session:
- **mean positive / mean negative** - average bias amplitude in each direction
- **peak positive / peak negative** - most extreme values
- **mean absolute** - overall magnitude of bias regardless of direction
- **variability** - mean absolute change between consecutive trials
- **slope** - linear trend across the session

These TL-BS parameters are compared against traditional session-level means as classification features. After computing per-session TL-BS parameters and traditional aggregates, features are further averaged per user to reduce within-subject measurement noise.

Three tasks are tested:
- binary classification (depressed vs not)
- multi-class (severity groups)
- regression (predict score directly)

Every task is run on two depression scales:
- **PHQ-9** (Patient Health Questionnaire, 9 items, range 0–27)
- **BDI-II** (Beck Depression Inventory, 21 items, range 0–63)

Running both scales provides a convergent-validity check: if the same gaze features predict both questionnaires similarly, the signal reflects the depression construct rather than questionnaire-specific content.

Three models are compared:
- Logistic / Ridge Regression
- Random Forest
- XGBoost

Across three feature sets:
- TL-BS only
- Traditional means
- Combined

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats as scipy_stats

from src.evaluation.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance,
)

## 1. Build session-level dataset

### 1.1 Load trial-level data and session-level means

In [0]:
scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = scene_metrics.filter(F.col("scene_type") == "stimulus")

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = df_stimulus.withColumn("trial_num", F.row_number().over(w))

df_trials = numbered.select("session_id", "trial_num", "fixation_bias").toPandas()

MEAN_METRICS = [
    "fixation_count", "mean_fixation_duration_ms", "fixation_rate_per_sec",
    "fixation_bias", "scanpath_length", "saccade_count",
    "blink_count", "blink_rate_per_min",
    "dwell_time_ms_negative", "dwell_time_ms_positive", "dwell_time_ms_neutral",
    "fixation_proportion_negative", "fixation_proportion_positive", "fixation_proportion_neutral",
    "revisit_count_negative", "revisit_count_positive",
    "ttff_ms_negative", "ttff_ms_positive",
    "first_fixation_duration_ms",
]

# Session-level traditional means
traditional_aggs = [F.mean(m).alias(f"avg_{m}") for m in MEAN_METRICS]
for valence in ["negative", "positive", "neutral"]:
    traditional_aggs.append(
        F.avg(F.when(F.col("first_fixation_valence") == valence, 1).otherwise(0))
         .alias(f"first_fix_prob_{valence}"))

df_traditional = df_stimulus.groupBy("session_id").agg(*traditional_aggs).toPandas()

print(f"Trial-level data: {len(df_trials)} rows, {df_trials['session_id'].nunique()} sessions")
print(f"Session-level aggregates: {len(df_traditional)} sessions")

### 1.2 Compute TL-BS parameters per session

Zvielli-style TL-BS parameters from the fixation_bias trial series:
- Positive bias = more attention to negative stimuli
- Negative bias = more attention to positive stimuli

In [0]:
def compute_tlbs_params(group):
    """
    Compute Zvielli-style TL-BS parameters from fixation_bias trial series.
    """
    values = group["fixation_bias"].dropna().values
    trial_nums = group.loc[group["fixation_bias"].notna(), "trial_num"].values

    if len(values) < 3:
        return {"tlbs_mean_pos": np.nan, "tlbs_mean_neg": np.nan,
                "tlbs_peak_pos": np.nan, "tlbs_peak_neg": np.nan,
                "tlbs_mean_abs": np.nan,
                "tlbs_variability": np.nan, "tlbs_slope": np.nan}

    pos_vals = values[values > 0]
    neg_vals = values[values < 0]

    slope, _, _, _, _ = scipy_stats.linregress(trial_nums, values)

    return {
        "tlbs_mean_pos":    float(np.mean(pos_vals)) if len(pos_vals) > 0 else 0.0,
        "tlbs_mean_neg":    float(np.mean(neg_vals)) if len(neg_vals) > 0 else 0.0,
        "tlbs_peak_pos":    float(np.max(values)),
        "tlbs_peak_neg":    float(np.min(values)),
        "tlbs_mean_abs":    float(np.mean(np.abs(values))),
        "tlbs_variability": float(np.mean(np.abs(np.diff(values)))),
        "tlbs_slope":       float(slope),
    }

tlbs_rows = []
for session_id, group in df_trials.groupby("session_id"):
    group = group.sort_values("trial_num")
    row = {"session_id": session_id}
    row.update(compute_tlbs_params(group))
    tlbs_rows.append(row)

df_tlbs = pd.DataFrame(tlbs_rows)
print(f"TL-BS features computed for {len(df_tlbs)} sessions (7 parameters each)")

### 1.3 Visualize trial-level bias signal by severity

In [0]:
df_forms_viz = df_forms.select("session_id", "phq9_score").toPandas()

def _viz_severity(s):
    if s <= 4:  return "minimal"
    if s <= 14: return "moderate"
    return "severe"

df_forms_viz["viz_severity"] = df_forms_viz["phq9_score"].apply(_viz_severity)
df_viz = df_trials.merge(df_forms_viz, on="session_id")

fig, axes = plt.subplots(1, 3, figsize=(20, 5), sharey=True)

for ax, sev in zip(axes, ["minimal", "moderate", "severe"]):
    subset = df_viz[df_viz["viz_severity"] == sev]
    sample_sessions = subset["session_id"].unique()[:5]
    for sid in sample_sessions:
        vals = subset[subset["session_id"] == sid].sort_values("trial_num")["fixation_bias"].values
        ax.plot(range(len(vals)), vals, alpha=0.5, linewidth=1.5)
    ax.axhline(y=0, color="black", linewidth=0.8, linestyle="--")
    ax.set_title(f"PHQ-9: {sev}")
    ax.set_xlabel("Trial number")
    ax.set_ylabel("Fixation bias" if ax == axes[0] else "")

plt.suptitle("Trial-level fixation bias signal", fontsize=13)
plt.tight_layout()
plt.show()

### 1.4 Merge features and aggregate to user level

TL-BS parameters + traditional means are merged with the forms table, then averaged per user.

In [0]:
df = df_tlbs.merge(df_traditional, on="session_id")
df = df.merge(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score").toPandas(),
    on="session_id",
)

FEATURE_COLS = [c for c in df.columns if c.startswith(("tlbs_", "avg_", "first_fix_prob_"))]
NUMERIC_COLS = FEATURE_COLS + ["phq9_score", "bdi_score"]

df = df.groupby("uid", as_index=False)[NUMERIC_COLS].mean()

print(f"After per-user aggregation: {len(df)} users")
print(f"PHQ-9: min={df['phq9_score'].min():.1f}, max={df['phq9_score'].max():.1f}, mean={df['phq9_score'].mean():.1f}")
print(f"BDI-II: min={df['bdi_score'].min():.1f}, max={df['bdi_score'].max():.1f}, mean={df['bdi_score'].mean():.1f}")

## 2. Define targets and feature sets

### 2.1 PHQ-9 targets

Binary cutoff: PHQ-9 ≥ 10 (standard Kroenke clinical threshold).
Severity classes (Kroenke et al., 2001):
- 0 = minimal (0–4)
- 1 = mild (5–9)
- 2 = moderate (10–14)
- 3 = moderately severe (15–19)
- 4 = severe (20–27)

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)

def phq9_severity_from_score(s):
    if s <= 4:  return 0   # minimal
    if s <= 9:  return 1   # mild
    if s <= 14: return 2   # moderate
    if s <= 19: return 3   # moderately severe
    return 4               # severe

df["phq9_severity_class"] = df["phq9_score"].apply(phq9_severity_from_score)

print("PHQ-9")
print(f"  Binary (>=10): {df['phq9_depressed'].value_counts().to_dict()}")
print(f"  Multi-class:   {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

### 2.2 BDI-II targets

Binary cutoff: BDI-II ≥ 14 (Beck et al., 1996 threshold for mild or worse depression; 81% sensitivity, 92% specificity against DSM-IV MDD).
Severity classes (Beck et al., 1996):
- 0 = minimal (0–13)
- 1 = mild (14–19)
- 2 = moderate (20–28)
- 3 = severe (29–63)

In [0]:
df["bdi_depressed"] = (df["bdi_score"] >= 14).astype(int)

def bdi_severity_from_score(s):
    if s <= 13: return 0   # minimal
    if s <= 19: return 1   # mild
    if s <= 28: return 2   # moderate
    return 3               # severe

df["bdi_severity_class"] = df["bdi_score"].apply(bdi_severity_from_score)

print("BDI-II")
print(f"  Binary (>=14): {df['bdi_depressed'].value_counts().to_dict()}")
print(f"  Multi-class:   {df['bdi_severity_class'].value_counts().sort_index().to_dict()}")

### 2.3 Feature sets

In [0]:
TLBS_FEATURES = [
    "tlbs_mean_pos", "tlbs_mean_neg",
    "tlbs_peak_pos", "tlbs_peak_neg",
    "tlbs_mean_abs",
    "tlbs_variability", "tlbs_slope",
]

TRADITIONAL_FEATURES = [f"avg_{m}" for m in MEAN_METRICS] + [
    "first_fix_prob_negative", "first_fix_prob_positive", "first_fix_prob_neutral",
]

COMBINED_FEATURES = TRADITIONAL_FEATURES + TLBS_FEATURES

FEATURE_SETS = {
    "TL-BS Only":        TLBS_FEATURES,
    "Traditional Means": TRADITIONAL_FEATURES,
    "Combined":          COMBINED_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = [
    "phq9_score", "phq9_depressed", "phq9_severity_class",
    "bdi_score",  "bdi_depressed",  "bdi_severity_class",
]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. TL-BS correlations with depression

Spearman correlations between each TL-BS parameter and both depression scales. Reference line shows correlation of the session-averaged fixation bias for comparison.

### 4.1 TL-BS correlations with PHQ-9

In [0]:
print("TL-BS parameter correlations with PHQ-9:\n")
for feat in TLBS_FEATURES:
    r, p = scipy_stats.spearmanr(df_clean[feat], df_clean["phq9_score"])
    star = "*" if p < 0.05 else " "
    print(f"  {star} r={r:+.3f}  p={p:.2e}  {feat}")

r_trad, p_trad = scipy_stats.spearmanr(df_clean["avg_fixation_bias"], df_clean["phq9_score"])
print()
print(f"Reference: avg_fixation_bias  r={r_trad:+.3f}  p={p_trad:.2e}")

### 4.2 TL-BS correlations with BDI-II

In [0]:
print("TL-BS parameter correlations with BDI-II:\n")
for feat in TLBS_FEATURES:
    r, p = scipy_stats.spearmanr(df_clean[feat], df_clean["bdi_score"])
    star = "*" if p < 0.05 else " "
    print(f"  {star} r={r:+.3f}  p={p:.2e}  {feat}")

r_trad, p_trad = scipy_stats.spearmanr(df_clean["avg_fixation_bias"], df_clean["bdi_score"])
print()
print(f"Reference: avg_fixation_bias  r={r_trad:+.3f}  p={p_trad:.2e}")

## 5. Binary classification

Four binary analyses crossing two depression scales with two thresholding strategies:
- clinical cutoff (PHQ-9 ≥ 10, BDI-II ≥ 14)
- extremes task (drop the ambiguous middle; compare minimal vs severe)


### 5.1 PHQ-9 cutoff (≥ 10)

#### 5.1.1 Run classification

In [0]:
y_phq9_binary = df_clean["phq9_depressed"].values
phq9_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups)

#### 5.1.2 Results

In [0]:
print(phq9_binary_df.to_string(index=False))

#### 5.1.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups, phq9_binary_df)

### 5.2 PHQ-9 extremes (minimal vs severe)

Minimal (PHQ-9 ≤ 4) vs Severe (PHQ-9 ≥ 20). Users with intermediate scores are excluded.

#### 5.2.1 Run classification

In [0]:
df_phq9_extreme = df_clean[(df_clean["phq9_score"] <= 4) | (df_clean["phq9_score"] >= 20)].copy()
df_phq9_extreme["phq9_extreme"] = (df_phq9_extreme["phq9_score"] >= 20).astype(int)

groups_phq9_extreme = df_phq9_extreme["uid"].values

print(f"PHQ-9 extremes dataset: {len(df_phq9_extreme)} users")
print(f"  Minimal (<=4):  {(df_phq9_extreme['phq9_extreme'] == 0).sum()}")
print(f"  Severe  (>=20): {(df_phq9_extreme['phq9_extreme'] == 1).sum()}")

y_phq9_extreme = df_phq9_extreme["phq9_extreme"].values
phq9_extreme_df = run_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme)

#### 5.2.2 Results

In [0]:
print(phq9_extreme_df.to_string(index=False))

#### 5.2.3 Best model

In [0]:
plot_best_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme, phq9_extreme_df)

### 5.3 BDI-II cutoff (≥ 14)

#### 5.3.1 Run classification

In [0]:
y_bdi_binary = df_clean["bdi_depressed"].values
bdi_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups)

#### 5.3.2 Results

In [0]:
print(bdi_binary_df.to_string(index=False))

#### 5.3.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups, bdi_binary_df)

### 5.4 BDI-II extremes (minimal vs severe)

Minimal (BDI ≤ 13) vs Severe (BDI ≥ 29). Users with intermediate scores are excluded.

#### 5.4.1 Run classification

In [0]:
df_bdi_extreme = df_clean[(df_clean["bdi_score"] <= 13) | (df_clean["bdi_score"] >= 29)].copy()
df_bdi_extreme["bdi_extreme"] = (df_bdi_extreme["bdi_score"] >= 29).astype(int)

groups_bdi_extreme = df_bdi_extreme["uid"].values

print(f"BDI-II extremes dataset: {len(df_bdi_extreme)} users")
print(f"  Minimal (<=13): {(df_bdi_extreme['bdi_extreme'] == 0).sum()}")
print(f"  Severe  (>=29): {(df_bdi_extreme['bdi_extreme'] == 1).sum()}")

y_bdi_extreme = df_bdi_extreme["bdi_extreme"].values
bdi_extreme_df = run_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme)

#### 5.4.2 Results

In [0]:
print(bdi_extreme_df.to_string(index=False))

#### 5.4.3 Best model

In [0]:
plot_best_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme, bdi_extreme_df)

## 6. Multi-class classification

### 6.1 PHQ-9 (5 severity classes)

#### 6.1.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_phq9_multi = df_clean["phq9_severity_class"].values.astype(int)
phq9_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups)

#### 6.1.2 Results

In [0]:
print(phq9_multi_df.to_string(index=False))

#### 6.1.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups, phq9_multi_df, PHQ9_LABELS)

### 6.2 BDI-II (4 severity classes)

#### 6.2.1 Run classification

In [0]:
BDI_LABELS = ["Minimal", "Mild", "Moderate", "Severe"]
y_bdi_multi = df_clean["bdi_severity_class"].values.astype(int)
bdi_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups)

#### 6.2.2 Results

In [0]:
print(bdi_multi_df.to_string(index=False))

#### 6.2.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups, bdi_multi_df, BDI_LABELS)

## 7. Regression

### 7.1 PHQ-9 score prediction

#### 7.1.1 Run regression

In [0]:
y_phq9_reg = df_clean["phq9_score"].values
phq9_reg_df = run_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups)

#### 7.1.2 Results

In [0]:
print(phq9_reg_df.to_string(index=False))

#### 7.1.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups, phq9_reg_df, score_name="PHQ-9")

### 7.2 BDI-II score prediction

#### 7.2.1 Run regression

In [0]:
y_bdi_reg = df_clean["bdi_score"].values
bdi_reg_df = run_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups)

#### 7.2.2 Results

In [0]:
print(bdi_reg_df.to_string(index=False))

#### 7.2.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups, bdi_reg_df, score_name="BDI-II")

## 8. PHQ-9 vs BDI-II convergent-validity comparison

Side-by-side best performance across all tasks. If the two scales produce similar numbers, the gaze-based depression signal generalizes across questionnaires.

In [0]:
def _best_auc(df_res):
    return df_res.sort_values("auc_roc", ascending=False).iloc[0]

def _best_f1(df_res):
    return df_res.sort_values("f1_weighted", ascending=False).iloc[0]

def _best_r2(df_res):
    return df_res.sort_values("r2", ascending=False).iloc[0]

rows = []

p = _best_auc(phq9_binary_df)
b = _best_auc(bdi_binary_df)
rows.append({"task": "Binary cutoff", "phq9_metric": "AUC",  "phq9_value": p["auc_roc"], "phq9_model": f"{p['model']} / {p['feature_set']}", "bdi_metric": "AUC",   "bdi_value": b["auc_roc"], "bdi_model": f"{b['model']} / {b['feature_set']}"})

p = _best_auc(phq9_extreme_df)
b = _best_auc(bdi_extreme_df)
rows.append({"task": "Binary extremes", "phq9_metric": "AUC",  "phq9_value": p["auc_roc"], "phq9_model": f"{p['model']} / {p['feature_set']}", "bdi_metric": "AUC",   "bdi_value": b["auc_roc"], "bdi_model": f"{b['model']} / {b['feature_set']}"})

p = _best_f1(phq9_multi_df)
b = _best_f1(bdi_multi_df)
rows.append({"task": "Multi-class", "phq9_metric": "F1",   "phq9_value": p["f1_weighted"], "phq9_model": f"{p['model']} / {p['feature_set']}", "bdi_metric": "F1",    "bdi_value": b["f1_weighted"], "bdi_model": f"{b['model']} / {b['feature_set']}"})

p = _best_r2(phq9_reg_df)
b = _best_r2(bdi_reg_df)
rows.append({"task": "Regression", "phq9_metric": "R2",   "phq9_value": p["r2"], "phq9_model": f"{p['model']} / {p['feature_set']}", "bdi_metric": "R2",    "bdi_value": b["r2"], "bdi_model": f"{b['model']} / {b['feature_set']}"})

comparison_df = pd.DataFrame(rows)
print(comparison_df.to_string(index=False))

## 9. Feature importance

Feature importance on the PHQ-9 binary task (Random Forest, combined features). Presented once because PHQ-9 and BDI-II converge on similar predictors.

In [0]:
plot_feature_importance(df_clean, COMBINED_FEATURES, y_phq9_binary, title="Feature importance (PHQ-9 binary, combined)")

## 10. Summary

### 10.1 PHQ-9

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(phq9_binary_df, phq9_multi_df, phq9_reg_df, feature_order, title="PHQ-9 — TL-BS features")

### 10.2 BDI-II

In [0]:
plot_summary(bdi_binary_df, bdi_multi_df, bdi_reg_df, feature_order, title="BDI-II — TL-BS features")